# Phase 5 — Model Deployment (FastAPI)
## Adaptive Real-Time Financial Transaction Fraud Detection System

**Notebook:** `notebooks/05_model_deployment.ipynb`
**Inputs:** `../models/xgboost_model_v1.json`, `../models/model_v1_metadata.json`, `../models/model_v1_threshold.json`, `../reports/feature_engineering_summary.json` (Phase 3), `../reports/selected_features.csv` / `feature_dictionary.csv` / `feature_leakage_audit.csv` (Phase 3)
**Generates:** `../src/api/app.py` (+ two tiny `__init__.py` files so `src.api.app` is importable — see Section 12)

> **What this notebook does:** turns the trained, saved Model V1 into a running FastAPI service that accepts a raw transaction, reproduces the exact Phase 3 feature pipeline, and returns a fraud probability, risk score and risk level.
> **What it does NOT do:** Kafka, Spark, SHAP, drift detection, retraining, Model V2, a dashboard, or authentication. Those are later phases.

## SECTION 1 — Phase 5 Introduction

**Why deployment is a separate phase.** A trained model sitting inside a notebook (`model` as a Python variable in Phase 4's kernel) is not yet something anything else can use. The kernel is temporary; the variable disappears when it shuts down. Deployment turns the *artifact on disk* (`xgboost_model_v1.json`) into a *service* — a long-running process that anything (a website, a simulator, later Kafka/Spark) can call over HTTP at any time, without needing to know Python, pandas, or how the model was trained.

**Training vs. deployment, side by side**

| | Training (Phase 4) | Deployment (Phase 5) |
|---|---|---|
| Input | The whole historical dataset, already feature-engineered, at once | One new transaction at a time |
| Direction | `Historical data → Model` (fit parameters) | `New transaction → Model → Prediction` (apply fixed parameters) |
| Feature engineering | Batch, vectorised, can see the whole file (but still past-only per event — Phase 3) | Must be recomputed **per request**, using only that transaction plus state accumulated from transactions already seen |
| Runs | Once (until Phase 10 retrains) | Continuously, for as long as the service is running |
| Output | A saved model file + metadata | A JSON response per request |

**The one hard constraint carried over from Phase 3.** Model V1 was trained on an *exact* list of feature columns, in an *exact* order (`model_v1_metadata.json → feature_names`). At inference time we must reproduce **exactly** those columns, computed with **exactly** the Phase 3 formulas and frozen constants (`feature_engineering_summary.json → frozen_parameters_fitted_on_training_period`) — nothing invented, nothing reordered. Sections 4–13 exist specifically to prove that this reproduction is correct before any HTTP request is ever served.

## SECTION 2 — Environment Check

Checks (does not blindly reinstall) `fastapi`, `uvicorn`, `pydantic`, `requests`, plus the Phase 1–4 stack (`pandas`, `numpy`, `xgboost`, `scikit-learn`). Missing deployment-only packages are installed with `%pip install` (a Jupyter magic — never placed inside the generated `app.py`, which must run as a plain script).

In [1]:
import importlib
import subprocess
import sys

REQUIRED = {"pandas": "pandas", "numpy": "numpy", "xgboost": "xgboost", "sklearn": "scikit-learn",
           "fastapi": "fastapi", "uvicorn": "uvicorn", "pydantic": "pydantic", "requests": "requests"}
missing_pip_names = []
print("Environment check:")
for module_name, pip_name in REQUIRED.items():
    try:
        mod = importlib.import_module(module_name)
        print(f"  [OK] {pip_name:<14} {getattr(mod, '__version__', 'version n/a')}")
    except ImportError:
        print(f"  [MISSING] {pip_name}")
        missing_pip_names.append(pip_name)

if missing_pip_names:
    print("\nInstalling missing packages:", missing_pip_names)
else:
    print("\nAll required packages are already available; nothing to install.")

Environment check:
  [MISSING] pandas
  [OK] numpy          2.5.3
  [OK] xgboost        3.4.1
  [OK] scikit-learn   1.9.1
  [MISSING] fastapi
  [MISSING] uvicorn
  [MISSING] pydantic
  [MISSING] requests

Installing missing packages: ['pandas', 'fastapi', 'uvicorn', 'pydantic', 'requests']


In [2]:
# NOTE: this cell only runs %pip install if Section 2 found something missing.
if missing_pip_names:
    get_ipython().run_line_magic("pip", "install " + " ".join(missing_pip_names))
    for module_name in list(REQUIRED):
        importlib.reload(importlib.import_module(module_name)) if module_name in sys.modules else None
    print("Re-run Section 2's check cell above to confirm installation before continuing.")
else:
    print("Nothing to install.")

  Using cached pandas-3.0.6-cp312-cp312-win_amd64.whl.metadata (19 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached tzdata-2026.4-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached starlette-1.7.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached charset_normalizer-3.5.1-cp312-cp312-win_amd64.whl.metadata (46 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cached urllib3-2.8.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached certifi-2026.7.22-py3-none-any.whl.metadata (2.5 kB)
  Using cached anyio-4.15.1-py3-none-any.whl.metadata (4.7 kB)
Using cached pandas-3.0.6-cp312-cp312-win_amd64.whl (9.7 MB)
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   --------------- ------------------------ 0.8/2.0 MB 4.2 MB/s eta 0:00:01
   ------------------------- -------------- 1.3/2.0 MB 3.2 MB/


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
C:\Users\MYSHA MUBASHIR\anaconda3\Lib\importlib\__init__.py:131: UserWarning: The NumPy module was reloaded (imported a second time). This can in some cases result in small but subtle issues and is discouraged.
  _bootstrap._exec(spec, module)


In [3]:
import importlib.util
for module_name, pip_name in REQUIRED.items():
    assert importlib.util.find_spec(module_name) is not None, f"'{pip_name}' is still not available - install it and restart the kernel."
import fastapi
import pydantic
import uvicorn
print(f"fastapi {fastapi.__version__} | pydantic {pydantic.__version__} | uvicorn {uvicorn.__version__}")
print(f"pydantic major version: {pydantic.VERSION.split('.')[0]} (this notebook and the generated app.py use the pydantic v2 API)")
assert int(pydantic.VERSION.split(".")[0]) >= 2, "This notebook targets pydantic v2's API (field_validator, model_config)."

fastapi 0.141.1 | pydantic 2.13.5 | uvicorn 0.54.0
pydantic major version: 2 (this notebook and the generated app.py use the pydantic v2 API)


## SECTION 3 — Project Paths

Relative paths, resolved from this notebook's location (`notebooks/`) so the notebook works regardless of the current working directory Jupyter happened to start in.

In [4]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from xgboost import XGBClassifier

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

MODEL_PATH = Path("../models/xgboost_model_v1.json")
METADATA_PATH = Path("../models/model_v1_metadata.json")
THRESHOLD_PATH = Path("../models/model_v1_threshold.json")
PROCESSED_DATA_PATH = Path("../data/processed/featured_transactions.parquet")
FALLBACK_PROCESSED_PATH = Path("../data/processed/featured_transactions.csv.gz")
RAW_DATA_PATH = Path("../data/raw/PS_20174392719_1491204439457_log.csv")
FE_SUMMARY_PATH = Path("../reports/feature_engineering_summary.json")           # Phase 3: frozen feature-engineering constants
SELECTED_FEATURES_PATH = Path("../reports/selected_features.csv")               # Phase 3
FEATURE_DICT_PATH = Path("../reports/feature_dictionary.csv")                   # Phase 3
LEAKAGE_AUDIT_PATH = Path("../reports/feature_leakage_audit.csv")               # Phase 3

SRC_DIR = Path("../src")
API_DIR = SRC_DIR / "api"
APP_PATH = API_DIR / "app.py"
REPORTS_DIR = Path("../reports")
LOG_PATH = REPORTS_DIR / "model_v1_predictions_log.jsonl"

for p in (MODEL_PATH, METADATA_PATH, THRESHOLD_PATH, FE_SUMMARY_PATH, RAW_DATA_PATH):
    print(f"{'[OK]' if p.is_file() else '[MISSING]'} {p}  ->  {p.resolve()}")
assert MODEL_PATH.is_file(), "Phase 4 model artifact not found - run 04_xgboost_model_v1.ipynb first."
assert METADATA_PATH.is_file() and THRESHOLD_PATH.is_file(), "Phase 4 metadata/threshold artifacts not found."
assert FE_SUMMARY_PATH.is_file(), ("Phase 3 feature-engineering summary not found. Phase 5 needs the frozen feature "
                                   "parameters (large-amount threshold, type categories, velocity windows) it contains "
                                   "to reproduce the exact Phase 3 feature definitions at inference time.")
print("\nProject root (resolved):", PROJECT_ROOT)

[OK] ..\models\xgboost_model_v1.json  ->  D:\fraud-detection-system\fraud-detection-system\models\xgboost_model_v1.json
[OK] ..\models\model_v1_metadata.json  ->  D:\fraud-detection-system\fraud-detection-system\models\model_v1_metadata.json
[OK] ..\models\model_v1_threshold.json  ->  D:\fraud-detection-system\fraud-detection-system\models\model_v1_threshold.json
[OK] ..\reports\feature_engineering_summary.json  ->  D:\fraud-detection-system\fraud-detection-system\reports\feature_engineering_summary.json
[OK] ..\data\raw\PS_20174392719_1491204439457_log.csv  ->  D:\fraud-detection-system\fraud-detection-system\data\raw\PS_20174392719_1491204439457_log.csv

Project root (resolved): D:\fraud-detection-system\fraud-detection-system


## SECTION 4 — Load Model V1

Loaded read-only for inspection/validation in this notebook. **The model is not retrained.**

In [5]:
model = XGBClassifier()
model.load_model(str(MODEL_PATH))
print("Loaded Model V1 from:", MODEL_PATH)
print("XGBoost version (this environment):", xgb.__version__)
booster_feature_names = model.get_booster().feature_names
print("Booster reports", len(booster_feature_names), "feature names (used for the compatibility check in Section 7).")

Loaded Model V1 from: ..\models\xgboost_model_v1.json
XGBoost version (this environment): 3.4.1
Booster reports 34 feature names (used for the compatibility check in Section 7).


## SECTION 5 — Load Model Metadata

`model_v1_metadata.json` (Phase 4) and `model_v1_threshold.json` (Phase 4) are the two authoritative sources for "what does Model V1 expect and how should its output be turned into a decision". Nothing here is hardcoded independently of these files.

In [6]:
with open(METADATA_PATH, encoding="utf-8") as fh:
    METADATA = json.load(fh)
with open(THRESHOLD_PATH, encoding="utf-8") as fh:
    THRESHOLD_INFO = json.load(fh)
with open(FE_SUMMARY_PATH, encoding="utf-8") as fh:
    FE_SUMMARY = json.load(fh)

FINAL_FEATURES = METADATA["feature_names"]                 # exact names, exact order
MODEL_VERSION = METADATA["model_version"]
TARGET = METADATA["target"]
SELECTED_THRESHOLD = THRESHOLD_INFO["selected_threshold"]
RISK_LEVELS = THRESHOLD_INFO["risk_levels"]
FE_PARAMS = FE_SUMMARY["frozen_parameters_fitted_on_training_period"]           # Phase 3 frozen constants

print("Model               :", METADATA["model_name"])
print("Model version       :", MODEL_VERSION)
print("Target              :", TARGET)
print("Expected feature count:", len(FINAL_FEATURES))
print("Classification threshold:", SELECTED_THRESHOLD, "  (reason:", THRESHOLD_INFO["reason"][:90] + "...)")
print("Risk levels         :", RISK_LEVELS)
print("\nFrozen feature-engineering parameters (from Phase 3):")
for k, v in FE_PARAMS.items():
    print(f"  {k:<24}: {v}")
assert TARGET == "isFraud"

Model               : fraud_detection_xgboost
Model version       : V1
Target              : isFraud
Expected feature count: 34
Classification threshold: 0.6   (reason: Threshold 0.60 maximises F1 = 0.9369 on the VALIDATION split (precision=0.8812, recall=1.0...)
Risk levels         : {'LOW': [0, 30], 'MEDIUM': [30, 70], 'HIGH': [70, 100]}

Frozen feature-engineering parameters (from Phase 3):
  large_amount_threshold  : 1452542.9963999956
  type_categories         : ['CASH_IN', 'CASH_OUT', 'DEBIT', 'PAYMENT', 'TRANSFER']
  tolerance               : 0.01
  windows                 : [1, 24, 168]
  sum_windows             : [24, 168]
  newcp_windows           : [24, 168]
  hours_per_day           : 24
  retain_flag_feature     : False
  train_end_step          : 323
  train_fraction          : 0.7


## SECTION 6 — Model Load Validation

A deployment smoke test using one **real row from the Phase 3 processed test data** — i.e. already in the exact engineered-feature form Model V1 was trained on. This checks that the loaded model object itself works; it does **not** yet test the raw-transaction → feature pipeline (that is Section 13). This probability is a sanity check only and is not logged as a real prediction.

In [7]:
def load_processed_sample():
    if PROCESSED_DATA_PATH.is_file():
        return pd.read_parquet(PROCESSED_DATA_PATH, columns=FINAL_FEATURES + [TARGET])
    if FALLBACK_PROCESSED_PATH.is_file():
        return pd.read_csv(FALLBACK_PROCESSED_PATH, usecols=FINAL_FEATURES + [TARGET])
    raise FileNotFoundError("Neither the Parquet nor the CSV Phase 3 output was found; run 03_feature_engineering.ipynb.")


processed_sample_full = load_processed_sample()
sample_row = processed_sample_full.iloc[[-1]]                      # a real, already-engineered row
sample_X = sample_row[FINAL_FEATURES]
sample_proba = float(model.predict_proba(sample_X)[:, 1][0])

print("Model loaded successfully.")
print(f"Prediction probability : {sample_proba:.6f}   (on one real processed row, NOT a live prediction)")
print(f"Prediction threshold   : {SELECTED_THRESHOLD}")
print(f"(For reference only) actual label of this sampled row: {int(sample_row[TARGET].iloc[0])}")
assert 0.0 <= sample_proba <= 1.0
del processed_sample_full

Model loaded successfully.
Prediction probability : 0.631356   (on one real processed row, NOT a live prediction)
Prediction threshold   : 0.6
(For reference only) actual label of this sampled row: 1


## SECTION 7 — Inspect Expected Features

Cross-checking the model's expected feature list against Phase 3's own records, so the deployment pipeline is built from **inspected facts**, not assumptions.

In [8]:
feature_dictionary = pd.read_csv(FEATURE_DICT_PATH) if FEATURE_DICT_PATH.is_file() else pd.DataFrame()
selected_features_report = pd.read_csv(SELECTED_FEATURES_PATH) if SELECTED_FEATURES_PATH.is_file() else pd.DataFrame()
leakage_audit_report = pd.read_csv(LEAKAGE_AUDIT_PATH) if LEAKAGE_AUDIT_PATH.is_file() else pd.DataFrame()

if len(selected_features_report):
    approved = set(selected_features_report["Feature"])
    mismatch_missing = [f for f in FINAL_FEATURES if f not in approved]
    mismatch_extra = [f for f in approved if f not in FINAL_FEATURES]
    print("Cross-check against reports/selected_features.csv (Phase 3):")
    print("  Model features NOT in Phase 3's approved list:", mismatch_missing if mismatch_missing else "none")
    print("  Phase 3 approved features NOT used by the model:", mismatch_extra if mismatch_extra else "none")
    assert not mismatch_missing, "Model expects a feature Phase 3 never approved - refusing to guess its definition."

if len(leakage_audit_report):
    unsafe_in_model = leakage_audit_report[(leakage_audit_report["feature_name"].isin(FINAL_FEATURES)) & (leakage_audit_report["leakage_status"] == "UNSAFE")]
    assert unsafe_in_model.empty, f"Model expects UNSAFE (leakage) feature(s): {unsafe_in_model['feature_name'].tolist()}"
    print("\nLeakage-audit cross-check: no UNSAFE feature is present in the model's feature list. OK.")

group_lookup = {}
if len(feature_dictionary):
    group_lookup = feature_dictionary.set_index("Feature")["Group"].to_dict()

groups_used = pd.Series([group_lookup.get(f, "Unknown (not in feature_dictionary.csv)") for f in FINAL_FEATURES]).value_counts()
print("\nFeature groups Model V1 actually uses (from the metadata's exact feature list):")
print(groups_used.to_string())

needs_history = [f for f in FINAL_FEATURES if group_lookup.get(f) in ("Account behaviour", "Velocity")]
single_txn_ok = [f for f in FINAL_FEATURES if f not in needs_history]
print(f"\nFeatures computable from the CURRENT transaction alone: {len(single_txn_ok)}")
print(f"Features that REQUIRE historical/account state       : {len(needs_history)}")
if needs_history:
    print("  ->", needs_history)
print("\nExact feature order expected by Model V1:")
print(FINAL_FEATURES)

Cross-check against reports/selected_features.csv (Phase 3):
  Model features NOT in Phase 3's approved list: none
  Phase 3 approved features NOT used by the model: none

Leakage-audit cross-check: no UNSAFE feature is present in the model's feature list. OK.

Feature groups Model V1 actually uses (from the metadata's exact feature list):
Account behaviour      10
Amount                  7
Velocity                7
Transaction type        5
Sender balance          2
Destination balance     2
Temporal                1

Features computable from the CURRENT transaction alone: 17
Features that REQUIRE historical/account state       : 17
  -> ['sender_prev_txn_count', 'sender_prev_total_amount', 'sender_prev_mean_amount', 'sender_prev_max_amount', 'dest_prev_txn_count', 'dest_prev_total_amount', 'dest_prev_mean_amount', 'dest_prev_max_amount', 'amount_vs_sender_hist_mean', 'amount_vs_dest_hist_mean', 'sender_hours_since_prev_txn', 'dest_hours_since_prev_txn', 'dest_txn_count_last_1h', 'des

## SECTION 8 — Build Inference Feature Pipeline

These functions are the **inference-time equivalent** of Phase 3's `create_temporal_features`, `create_amount_features`, `create_balance_features` and `encode_transaction_type` — same formulas, same frozen constants (`FE_PARAMS`, loaded in Section 5), but written to work on **one transaction's scalar values** instead of a whole DataFrame column.

Only **SAFE** and **POTENTIAL_RISK** features (per the Phase 3 leakage audit) are computed. `newbalanceOrig` / `newbalanceDest` never appear on the right-hand side of any formula below — the UNSAFE features Phase 3 excluded are simply never reconstructed here.

This code is assembled into a string now and **written into `src/api/app.py` unchanged** in Section 12 — there is exactly one copy of this logic, so the notebook and the running API can never drift apart. It is validated for real in Section 13, after being written to disk and imported back.

In [9]:
APP_FEATURE_FUNCS_SRC = '''
def safe_divide(numerator: float, denominator: float) -> float:
    """Returns NaN (not inf, not 0) when the denominator is not strictly positive."""
    if denominator is None or denominator <= 0 or numerator is None:
        return float("nan")
    return numerator / denominator


def create_temporal_features(step: int) -> dict:
    hour = (step - 1) % HOURS_PER_DAY
    return {
        "hour_of_day": hour,
        "hour_sin": math.sin(2 * math.pi * hour / HOURS_PER_DAY),
        "hour_cos": math.cos(2 * math.pi * hour / HOURS_PER_DAY),
        "time_period_bin": hour // 6,
        "day_index": (step - 1) // HOURS_PER_DAY,
    }


def create_amount_features(amount: float, old_balance_orig: float, old_balance_dest: float) -> dict:
    return {
        "amount": amount,
        "log_amount": math.log1p(max(amount, 0.0)),
        "is_zero_amount": int(amount == 0),
        "is_large_amount": int(amount >= LARGE_AMOUNT_THRESHOLD),
        "amount_to_sender_balance_ratio": safe_divide(amount, old_balance_orig),
        "amount_ge_sender_balance": int((amount >= old_balance_orig - FE_PARAMS["tolerance"]) and old_balance_orig > 0),
        "sender_remaining_if_debited": old_balance_orig - amount,
        "amount_to_dest_balance_ratio": safe_divide(amount, old_balance_dest),
    }


def create_balance_features(old_balance_orig: float, old_balance_dest: float) -> dict:
    """PRE-transaction balance features only. newbalanceOrig/newbalanceDest are never used here."""
    return {
        "sender_balance_before": old_balance_orig,
        "sender_balance_zero_before": int(old_balance_orig == 0),
        "dest_balance_before": old_balance_dest,
        "dest_balance_zero_before": int(old_balance_dest == 0),
    }


def encode_transaction_type(txn_type: str) -> dict:
    """One-hot encoding using the frozen category list fitted in Phase 3."""
    return {f"type_{cat}": int(txn_type == cat) for cat in TYPE_CATEGORIES}
'''
print(f"Feature-function source block ready ({len(APP_FEATURE_FUNCS_SRC.splitlines())} lines).")

Feature-function source block ready (45 lines).


## SECTION 9 — Build Historical State

`nameOrig` / `nameDest` history and velocity features (Phase 3, Section 7's registry) cannot be computed from a single transaction — they need what happened *before* it. Section 7 already listed exactly which of Model V1's features fall into this group (`needs_history`).

**This is the same algorithm as the `StreamingFeatureState` reference implementation built and verified in Phase 3, Section 21** (there it was proven to reproduce the batch account/velocity features exactly, one event at a time). Porting it here — rather than inventing a new approximation — is what "faithful reconstruction of the actual Phase 3 artifacts" means in practice.

**Explicitly a development/demo mechanism.** State lives in one Python dictionary inside one running process:
* it is **lost when the API restarts**;
* it does **not** synchronise across multiple API instances;
* it will be **replaced** by Kafka + Spark Structured Streaming stateful processing (keyed by account) in a later phase — the *feature definitions* stay identical, only *where the state lives* changes.

**Past-only guarantee:** for a transaction at step *t*, `compute_and_update` reads the account's state *before* touching it, and only appends the current event *after* the features are computed and returned — so a transaction can never see itself, or anything later, in its own history.

In [ ]:
APP_STATE_CLASS_SRC = '''
class AccountState:
    """
    In-memory, per-account historical state for the sender and destination roles.

    *** DEVELOPMENT / DEMO MECHANISM ONLY ***
    This is a plain Python dict living inside one running API process. It is intentionally simple
    for Phase 5: state is lost on restart, and it does not scale across multiple API instances.
    A later phase (Kafka + Spark Structured Streaming, stateful processing keyed by account) will
    replace this with a durable, shared state store. The FEATURE DEFINITIONS implemented here are
    exactly what that future stateful job must also compute - only the storage mechanism will change.

    Guarantee: for a transaction at step t, only events strictly BEFORE this call are ever read.
    The state is updated with the current transaction only AFTER its features are computed.
    """

    def __init__(self):
        self._state = {"sender": {}, "dest": {}}
        self._pairs: set[tuple[str, str]] = set()

    def reset(self) -> None:
        """Clear process-local history before a deterministic replay or demo run."""
        self._state = {"sender": {}, "dest": {}}
        self._pairs = set()

    @staticmethod
    def _new_account() -> dict:
        return {"count": 0, "total": 0.0, "max": float("nan"), "last_step": None, "uniq": 0, "events": deque()}

    def _role_features(self, role: str, acct: dict, step: int) -> dict:
        while acct["events"] and acct["events"][0][0] <= step - MAX_WINDOW:
            acct["events"].popleft()
        feats = {
            f"{role}_prev_txn_count": acct["count"],
            f"{role}_prev_total_amount": acct["total"],
            f"{role}_prev_mean_amount": (acct["total"] / acct["count"]) if acct["count"] else float("nan"),
            f"{role}_prev_max_amount": acct["max"],
            f"{role}_prev_unique_counterparties": acct["uniq"],
            f"{role}_hours_since_prev_txn": (step - acct["last_step"]) if acct["count"] else float("nan"),
        }
        for w in WINDOWS:
            feats[f"{role}_txn_count_last_{w}h"] = sum(1 for s, _, _ in acct["events"] if s > step - w)
        for w in SUM_WINDOWS:
            feats[f"{role}_amount_sum_last_{w}h"] = sum(a for s, a, _ in acct["events"] if s > step - w)
        for w in NEWCP_WINDOWS:
            feats[f"{role}_new_counterparties_last_{w}h"] = sum(f for s, _, f in acct["events"] if s > step - w)
        return feats

    def compute_and_update(self, sender: str, dest: str, step: int, amount: float) -> dict:
        """Return past-only features for this event, THEN update the state with it."""
        is_new_pair = (sender, dest) not in self._pairs
        s_acct = self._state["sender"].setdefault(sender, self._new_account())
        d_acct = self._state["dest"].setdefault(dest, self._new_account())

        feats = {}
        feats.update(self._role_features("sender", s_acct, step))
        feats.update(self._role_features("dest", d_acct, step))
        feats["is_new_destination_for_sender"] = int(is_new_pair)
        feats["amount_vs_sender_hist_mean"] = safe_divide(amount, feats["sender_prev_mean_amount"])
        feats["amount_vs_dest_hist_mean"] = safe_divide(amount, feats["dest_prev_mean_amount"])

        for acct in (s_acct, d_acct):
            acct["count"] += 1
            acct["total"] += amount
            acct["max"] = amount if (isinstance(acct["max"], float) and math.isnan(acct["max"])) else max(acct["max"], amount)
            acct["last_step"] = step
            acct["uniq"] += int(is_new_pair)
            acct["events"].append((step, amount, int(is_new_pair)))
        self._pairs.add((sender, dest))
        return feats

    def preview_features(self, sender: str, dest: str, step: int, amount: float) -> dict:
        """Compute one feature state without consuming the event, for parity validation only."""
        from copy import deepcopy
        state, pairs = self._state, self._pairs
        self._state, self._pairs = deepcopy(state), set(pairs)
        try:
            return self.compute_and_update(sender, dest, step, amount)
        finally:
            self._state, self._pairs = state, pairs

    def account_count(self) -> dict:
        return {"sender_accounts_tracked": len(self._state["sender"]), "dest_accounts_tracked": len(self._state["dest"])}


# One shared, process-wide state instance (see AccountState docstring for its scope/limitations).
account_state = AccountState()
'''
print(f"AccountState source block ready ({len(APP_STATE_CLASS_SRC.splitlines())} lines).")

AccountState source block ready (72 lines).


### Assembling the feature vector and the prediction function

`build_feature_vector` combines the row-level functions (Section 8) and the state class (Section 9) into exactly the candidate feature set, then **selects and reorders** it to match `FINAL_FEATURES` from `model_v1_metadata.json` **exactly**. If the model ever expects a feature this pipeline does not compute, it raises immediately instead of silently reordering or dropping columns — this is the automated version of Section 5's feature-compatibility requirement, now enforced on every single request.

`predict_transaction` then applies Model V1, the Phase 4 classification threshold, and the Phase 4 risk-score/risk-level definitions — **unchanged** from Phase 4.

In [ ]:
APP_PREDICT_FUNCS_SRC = '''
def build_feature_vector(txn: "TransactionRequest", update_state: bool = True) -> pd.DataFrame:
    """Build the exact Model V1 feature vector, optionally without consuming AccountState."""
    candidate = {}
    candidate.update(create_temporal_features(txn.step))
    candidate.update(create_amount_features(txn.amount, txn.oldbalanceOrg, txn.oldbalanceDest))
    candidate.update(create_balance_features(txn.oldbalanceOrg, txn.oldbalanceDest))
    state_features = (account_state.compute_and_update(txn.nameOrig, txn.nameDest, txn.step, txn.amount)
                      if update_state else account_state.preview_features(txn.nameOrig, txn.nameDest, txn.step, txn.amount))
    candidate.update(state_features)
    candidate.update(encode_transaction_type(txn.type))
    if RETAIN_FLAG_FEATURE:
        candidate["is_flagged_by_rule"] = int(txn.isFlaggedFraud or 0)

    missing = [f for f in FINAL_FEATURES if f not in candidate]
    if missing:
        raise ValueError(f"Feature mismatch: the model expects feature(s) the pipeline did not compute: {missing}. "
                         "Refusing to guess - check that app.py matches the current Phase 3/4 artifacts.")
    ordered = {f: candidate[f] for f in FINAL_FEATURES}
    return pd.DataFrame([ordered], columns=FINAL_FEATURES)


def assign_risk_level(score: float) -> str:
    for level, (lo, hi) in RISK_LEVELS.items():
        if lo <= score < hi or (level == "HIGH" and score >= hi):
            return level
    return "HIGH"


def predict_transaction(txn: "TransactionRequest") -> dict:
    """The full scoring function used by POST /predict."""
    X = build_feature_vector(txn)
    fraud_probability = float(model.predict_proba(X)[:, 1][0])
    predicted_label = int(fraud_probability >= CLASSIFICATION_THRESHOLD)
    risk_score = round(fraud_probability * 100, 4)
    risk_level = assign_risk_level(risk_score)
    return {"prediction": predicted_label, "fraud_probability": fraud_probability, "risk_score": risk_score,
            "risk_level": risk_level, "model_version": MODEL_VERSION}


def log_prediction(result: dict) -> None:
    """Lightweight local JSONL prediction log. Logs outputs only, never raw labels."""
    record = {"timestamp_utc": datetime.now(timezone.utc).isoformat(), "model_version": result["model_version"],
              "fraud_probability": result["fraud_probability"], "predicted_label": result["prediction"],
              "risk_score": result["risk_score"], "risk_level": result["risk_level"]}
    try:
        LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
        with open(LOG_PATH, "a", encoding="utf-8") as fh:
            fh.write(json.dumps(record) + "\\n")
    except OSError as exc:
        logger.warning("Could not write prediction log: %s", exc)
'''
print(f"Prediction-function source block ready ({len(APP_PREDICT_FUNCS_SRC.splitlines())} lines).")

Prediction-function source block ready (54 lines).


## SECTION 10 — Build Pydantic Schemas

**Raw transaction input schema.** Fields are the actual PaySim raw columns the feature pipeline needs. Two deliberate design choices, both explained inline in the generated file:

1. **`isFraud` does not exist as a field at all**, and `model_config = {"extra": "forbid"}` makes Pydantic **reject** (HTTP 422) any request that includes it or any other unrecognised field — the true label being unavailable at prediction time is enforced by the schema itself, not just by convention.
2. **`newbalanceOrig` / `newbalanceDest` are accepted as optional fields** purely so a raw historical PaySim CSV row can be posted as-is (minus `isFraud`) for testing — but the feature pipeline (Section 8) never reads them, since Phase 3 marked every feature derived from them `UNSAFE`.

Validation rules (→ automatic HTTP 422 on failure, standard FastAPI/Pydantic behaviour): `amount ≥ 0`, `oldbalanceOrg ≥ 0`, `oldbalanceDest ≥ 0`, `step ≥ 1`, `type` must be one of the categories Phase 3 actually trained on (`TYPE_CATEGORIES`, loaded from the frozen Phase 3 parameters — not hardcoded here).

In [12]:
APP_SCHEMA_SRC = '''
class TransactionRequest(BaseModel):
    step: int = Field(..., ge=1, description="Simulated hour of the transaction (PaySim: 1 step = 1 hour).")
    type: str = Field(..., description=f"Transaction type. One of: {TYPE_CATEGORIES}")
    amount: float = Field(..., ge=0, description="Transaction amount. Must be >= 0.")
    nameOrig: str = Field(..., min_length=1, description="Sender account identifier.")
    oldbalanceOrg: float = Field(..., ge=0, description="Sender balance BEFORE the transaction. Must be >= 0.")
    nameDest: str = Field(..., min_length=1, description="Destination account identifier.")
    oldbalanceDest: float = Field(..., ge=0, description="Destination balance BEFORE the transaction. Must be >= 0.")
    isFlaggedFraud: Optional[int] = Field(0, ge=0, le=1, description="Existing rule-based flag (0 or 1). Optional.")
    newbalanceOrig: Optional[float] = Field(None, description="Accepted for raw-row compatibility ONLY - NOT used by the model (post-transaction/leakage field, see Phase 3 leakage audit).")
    newbalanceDest: Optional[float] = Field(None, description="Accepted for raw-row compatibility ONLY - NOT used by the model (post-transaction/leakage field, see Phase 3 leakage audit).")

    @field_validator("type")
    @classmethod
    def _type_must_be_supported(cls, v: str) -> str:
        if v not in TYPE_CATEGORIES:
            raise ValueError(f"type must be one of {TYPE_CATEGORIES}, got {v!r}")
        return v

    model_config = {
        "extra": "forbid",   # CRITICAL: explicitly REJECTS any extra field (e.g. isFraud) with a 422, rather than silently ignoring it
        "json_schema_extra": {"example": {
            "step": 5, "type": "TRANSFER", "amount": 1500.0, "nameOrig": "C1234567890", "oldbalanceOrg": 5000.0,
            "nameDest": "C9876543210", "oldbalanceDest": 0.0, "isFlaggedFraud": 0}}}


class PredictionResponse(BaseModel):
    prediction: int
    fraud_probability: float
    risk_score: float
    risk_level: str
    model_version: str
'''
print(f"Pydantic-schema source block ready ({len(APP_SCHEMA_SRC.splitlines())} lines).")
print("\nKey safety property: 'isFraud' is not a field of TransactionRequest, and extra='forbid' turns any attempt")
print("to send it into an automatic HTTP 422 - this is enforced by the schema, not just by omission.")

Pydantic-schema source block ready (33 lines).

Key safety property: 'isFraud' is not a field of TransactionRequest, and extra='forbid' turns any attempt
to send it into an automatic HTTP 422 - this is enforced by the schema, not just by omission.


## SECTION 11 — Build FastAPI Application

Two remaining pieces: the **module header** (imports, path resolution, one-time model/config loading — this is what guarantees the model is loaded **once**, at process start, not per request) and the **endpoints** (`GET /`, `GET /health`, `POST /predict`) that call the functions built in Sections 8–10.

**Why the model loads once.** `uvicorn` imports `src/api/app.py` exactly once when a worker process starts. Everything at module level — `model = XGBClassifier(); model.load_model(...)`, reading the three JSON artifacts, `account_state = AccountState()` — therefore runs exactly once, and every subsequent request reuses the same in-memory `model` object and the same `account_state`. Re-loading the model inside the `/predict` function body would be both slower (disk I/O on every request) and would silently reset any in-memory state.

**Error handling.** A `ValueError` from `build_feature_vector` (a genuine feature mismatch) becomes an HTTP 400 with the specific reason. Anything else unexpected is logged server-side and returned to the client as a generic HTTP 500 — internal details (paths, stack traces) are never sent to the caller.

In [13]:
APP_HEADER_SRC = '''"""
Financial Fraud Detection API — Model V1
=========================================

Generated by notebooks/05_model_deployment.ipynb (Phase 5). Do not edit by hand; re-run the
notebook if the feature pipeline or model artifacts change.

Pipeline implemented here:

    Raw transaction (HTTP request)
        -> Pydantic validation
        -> update/read per-account historical state (in-memory, dev-only - see AccountState)
        -> compute the same candidate features Phase 3 defines
        -> select the EXACT feature names/order Model V1 was trained on (model_v1_metadata.json)
        -> XGBoost Model V1 .predict_proba()
        -> fraud_probability, risk_score, risk_level, model_version

CRITICAL LEAKAGE GUARD: `isFraud` is never accepted as API input, and post-transaction fields
(`newbalanceOrig`, `newbalanceDest`) are accepted only for compatibility with raw PaySim CSV rows
used in testing/simulation - they are NEVER read by the feature pipeline below, because Phase 3's
leakage audit marked every feature derived from them UNSAFE and Model V1 was not trained on them.
"""

from __future__ import annotations

import json
import logging
import math
from collections import deque
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field, field_validator
from xgboost import XGBClassifier

logger = logging.getLogger("fraud_api")
logging.basicConfig(level=logging.INFO)

# ---------------------------------------------------------------------------------------------
# Project paths - resolved from this file's own location so the API works regardless of the
# working directory it is started from (src/api/app.py -> parents[2] is the project root).
# ---------------------------------------------------------------------------------------------
PROJECT_ROOT = Path(__file__).resolve().parents[2]
MODEL_PATH = PROJECT_ROOT / "models" / "xgboost_model_v1.json"
METADATA_PATH = PROJECT_ROOT / "models" / "model_v1_metadata.json"
THRESHOLD_PATH = PROJECT_ROOT / "models" / "model_v1_threshold.json"
FE_SUMMARY_PATH = PROJECT_ROOT / "reports" / "feature_engineering_summary.json"
LOG_PATH = PROJECT_ROOT / "reports" / "model_v1_predictions_log.jsonl"

# ---------------------------------------------------------------------------------------------
# Load model + frozen configuration ONCE, at import time (not per-request).
# uvicorn imports this module a single time when the worker process starts, so `model` and the
# constants below are built once and reused for every request that process handles.
# ---------------------------------------------------------------------------------------------
for _p in (MODEL_PATH, METADATA_PATH, THRESHOLD_PATH, FE_SUMMARY_PATH):
    if not _p.is_file():
        raise FileNotFoundError(f"Required artifact not found: {_p}. Run notebooks 03/04 (and 05) before starting the API.")

model = XGBClassifier()
model.load_model(str(MODEL_PATH))

with open(METADATA_PATH, encoding="utf-8") as _fh:
    METADATA = json.load(_fh)
with open(THRESHOLD_PATH, encoding="utf-8") as _fh:
    THRESHOLD_INFO = json.load(_fh)
with open(FE_SUMMARY_PATH, encoding="utf-8") as _fh:
    FE_SUMMARY = json.load(_fh)

FINAL_FEATURES: list[str] = METADATA["feature_names"]           # exact names, exact order
MODEL_VERSION: str = METADATA["model_version"]
CLASSIFICATION_THRESHOLD: float = THRESHOLD_INFO["selected_threshold"]
RISK_LEVELS: dict = THRESHOLD_INFO["risk_levels"]                # display bands, independent of CLASSIFICATION_THRESHOLD
FE_PARAMS: dict = FE_SUMMARY["frozen_parameters_fitted_on_training_period"]
TYPE_CATEGORIES: list[str] = FE_PARAMS["type_categories"]
WINDOWS = tuple(FE_PARAMS["windows"])
SUM_WINDOWS = tuple(FE_PARAMS["sum_windows"])
NEWCP_WINDOWS = tuple(FE_PARAMS["newcp_windows"])
HOURS_PER_DAY = FE_PARAMS["hours_per_day"]
MAX_WINDOW = max(WINDOWS)
LARGE_AMOUNT_THRESHOLD = FE_PARAMS["large_amount_threshold"]
RETAIN_FLAG_FEATURE = bool(FE_PARAMS.get("retain_flag_feature", False))

logger.info("Model V1 loaded. %d expected features. Classification threshold=%.2f",
           len(FINAL_FEATURES), CLASSIFICATION_THRESHOLD)

# ---------------------------------------------------------------------------------------------
# Feature engineering - mirrors notebooks/03_feature_engineering.ipynb EXACTLY (same formulas,
# same frozen constants). Only the features that are SAFE or POTENTIAL_RISK per the Phase 3
# leakage audit are computed here; nothing derived from newbalanceOrig/newbalanceDest exists
# below, by construction (those columns are simply never read).
# ---------------------------------------------------------------------------------------------
'''
print(f"Header source block ready ({len(APP_HEADER_SRC.splitlines())} lines).")

Header source block ready (95 lines).


In [ ]:
APP_ENDPOINTS_SRC = '''
# ---------------------------------------------------------------------------------------------
# FastAPI application
# ---------------------------------------------------------------------------------------------
app = FastAPI(title="Financial Fraud Detection API", description="Serves Model V1 fraud predictions for the "
             "Adaptive Real-Time Financial Transaction Fraud Detection System.", version=MODEL_VERSION)


@app.get("/")
def root():
    return {"service": "Financial Fraud Detection API", "status": "running", "model_version": MODEL_VERSION}


@app.get("/health")
def health():
    return {"status": "healthy", "model_loaded": model is not None, "model_version": MODEL_VERSION}


@app.post("/state/reset")
def reset_state():
    """Development-only reset for deterministic historical replay before Phase 6."""
    account_state.reset()
    return {"status": "reset", "account_state": account_state.account_count()}


@app.post("/state/warmup")
def warmup_state(transactions: list[TransactionRequest]):
    """Replay pre-live transactions without model prediction or logging."""
    for txn in transactions:
        account_state.compute_and_update(txn.nameOrig, txn.nameDest, txn.step, txn.amount)
    return {"status": "warmed", "processed_transactions": len(transactions),
            "account_state": account_state.account_count()}


@app.post("/state/preview")
def preview_state(txn: TransactionRequest):
    """Return the exact Model V1 feature vector without consuming the transaction."""
    row = build_feature_vector(txn, update_state=False).iloc[0]
    features = {name: (None if pd.isna(value) else float(value)) for name, value in row.items()}
    return {"features": features, "feature_names": FINAL_FEATURES}


@app.post("/predict", response_model=PredictionResponse)
def predict(txn: TransactionRequest):
    try:
        result = predict_transaction(txn)
    except ValueError as exc:
        raise HTTPException(status_code=400, detail=str(exc)) from exc
    except Exception as exc:
        logger.exception("Unexpected error during prediction")
        raise HTTPException(status_code=500, detail="Internal error while generating a prediction.") from exc
    log_prediction(result)
    return result
'''
print(f"Endpoints source block ready ({len(APP_ENDPOINTS_SRC.splitlines())} lines).")

Endpoints source block ready (30 lines).


## SECTION 12 — Generate `src/api/app.py`

Assembles every block above, in dependency order (header/config → feature functions → state → prediction functions → schemas → endpoints), and writes **one** file. Two empty `__init__.py` files are created only if missing, so `src.api.app` is importable as a package — this is exactly the kind of "small supporting file required for deployment" this phase is allowed to add; nothing else in the existing project structure is touched.

In [15]:
APP_SOURCE = (APP_HEADER_SRC + APP_FEATURE_FUNCS_SRC + "\n\n" + APP_STATE_CLASS_SRC + "\n\n"
             + APP_PREDICT_FUNCS_SRC + "\n\n# " + "-" * 95
             + "\n# Request schema. CRITICAL: `isFraud` does not appear here - the true label is never known at\n"
             "# prediction time. newbalanceOrig/newbalanceDest are accepted (optional) only so a raw historical\n"
             "# PaySim CSV row can be sent as-is for testing; the feature pipeline above never reads them.\n# "
             + "-" * 95 + "\n" + APP_SCHEMA_SRC + APP_ENDPOINTS_SRC)

API_DIR.mkdir(parents=True, exist_ok=True)
for init_file in (SRC_DIR / "__init__.py", API_DIR / "__init__.py"):
    if not init_file.is_file():
        init_file.write_text("", encoding="utf-8")
        print("Created ->", init_file)
    else:
        print("Already exists, left untouched ->", init_file)

APP_PATH.write_text(APP_SOURCE, encoding="utf-8")
print(f"\nWrote {APP_PATH}  ({len(APP_SOURCE):,} characters, {len(APP_SOURCE.splitlines())} lines)")

Created -> ..\src\__init__.py
Created -> ..\src\api\__init__.py

Wrote ..\src\api\app.py  (17,418 characters, 340 lines)


## SECTION 13 — Validate the Generated Application

Three independent checks, all against the **file actually written to disk** (not the in-memory strings above), before any HTTP request is attempted:

1. **Syntax check** (`py_compile`) — catches typos/assembly errors without running any code.
2. **Import check** — dynamically imports `src/api/app.py` as a fresh module (loading the real Model V1 in the process), catching missing artifacts or runtime errors at import time.
3. **Feature-compatibility check** — calls the imported module's own `build_feature_vector` on one synthetic-but-schema-valid transaction and confirms the resulting column list is **exactly** `FINAL_FEATURES`, in the **exact same order**, matching the booster's own `feature_names` from Section 4. Any mismatch stops the notebook here rather than being discovered later as a silent wrong prediction.

In [16]:
import py_compile

py_compile.compile(str(APP_PATH), doraise=True)
print("[PASS] Syntax check (py_compile)")

[PASS] Syntax check (py_compile)


In [17]:
import importlib.util as _ilu

_spec = _ilu.spec_from_file_location("generated_fraud_api_app", APP_PATH)
generated_app_module = _ilu.module_from_spec(_spec)
sys.modules[_spec.name] = generated_app_module          # required so pydantic v2 can resolve annotations
                                                         # (the generated file uses `from __future__ import annotations`,
                                                         # which needs the module registered in sys.modules to look up
                                                         # `Optional` etc. when it builds TransactionRequest's schema;
                                                         # this step is a quirk of THIS notebook's manual dynamic-import
                                                         # validation only - uvicorn's normal package import of
                                                         # `src.api.app` is unaffected and needs no such workaround)
_spec.loader.exec_module(generated_app_module)
print("[PASS] Import check - the generated app.py imports cleanly and Model V1 loaded inside it.")
print("  Model version reported by the module:", generated_app_module.MODEL_VERSION)
print("  Classification threshold reported   :", generated_app_module.CLASSIFICATION_THRESHOLD)

INFO:fraud_api:Model V1 loaded. 34 expected features. Classification threshold=0.60


[PASS] Import check - the generated app.py imports cleanly and Model V1 loaded inside it.
  Model version reported by the module: V1
  Classification threshold reported   : 0.6


In [18]:
_probe_txn = generated_app_module.TransactionRequest(step=1, type=FE_PARAMS["type_categories"][0], amount=100.0,
                                                     nameOrig="C_PROBE_SENDER", oldbalanceOrg=1000.0,
                                                     nameDest="C_PROBE_DEST", oldbalanceDest=0.0, isFlaggedFraud=0)
_probe_X = generated_app_module.build_feature_vector(_probe_txn)

names_match = list(_probe_X.columns) == FINAL_FEATURES == booster_feature_names
print("Computed feature columns match FINAL_FEATURES (name AND order):", list(_probe_X.columns) == FINAL_FEATURES)
print("FINAL_FEATURES match the booster's own feature_names (name AND order):", FINAL_FEATURES == booster_feature_names)
assert names_match, "Feature name/order mismatch between the generated pipeline, the metadata, and the booster."
print("[PASS] Feature-compatibility check - no missing, no extra, no reordered features.")

_probe_proba = float(generated_app_module.model.predict_proba(_probe_X)[:, 1][0])
print(f"\nProbe transaction (fresh, never-seen accounts) -> fraud probability = {_probe_proba:.6f} (sanity check only, not a real prediction)")
assert 0.0 <= _probe_proba <= 1.0

Computed feature columns match FINAL_FEATURES (name AND order): True
FINAL_FEATURES match the booster's own feature_names (name AND order): True
[PASS] Feature-compatibility check - no missing, no extra, no reordered features.

Probe transaction (fresh, never-seen accounts) -> fraud probability = 0.394446 (sanity check only, not a real prediction)


## SECTION 14 — Start API Instructions

**To run the service normally** (from the `fraud-detection-system/` project root — not from inside `notebooks/`):

```bash
cd fraud-detection-system
python -m uvicorn src.api.app:app --reload
```

* **Start:** the command above. `--reload` auto-restarts the server when `app.py` changes (development convenience only — omit it in any real deployment).
* **Stop:** `Ctrl+C` in the terminal running uvicorn.
* **Swagger UI:** open `http://127.0.0.1:8000/docs` in a browser while the server is running.
* **Health check:** `curl http://127.0.0.1:8000/health` (or open the URL in a browser).
* **Prediction:** `POST http://127.0.0.1:8000/predict` with a JSON body — easiest done from the Swagger UI's "Try it out" button, or with `requests.post(...)` as demonstrated below.

**For the automated tests in this notebook (Sections 15–18)**, the server is instead started as a background **subprocess** on a free local port and stopped automatically at the end of Section 19 — this lets the notebook verify the exact file just written without leaving a server running afterwards, and without `--reload`'s extra watcher process complicating clean shutdown. The command launched is otherwise identical: `python -m uvicorn src.api.app:app --host 127.0.0.1 --port <port>`.

In [19]:
import socket
import subprocess
import sys

import requests


def find_free_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]


def start_api_server(port, timeout_s=20):
    """Launch uvicorn as a subprocess (cwd = project root, so 'src.api.app' resolves) and wait until it answers."""
    proc = subprocess.Popen([sys.executable, "-m", "uvicorn", "src.api.app:app", "--host", "127.0.0.1", "--port", str(port)],
                            cwd=str(PROJECT_ROOT), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    base_url = f"http://127.0.0.1:{port}"
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            if requests.get(base_url + "/", timeout=1).status_code == 200:
                return proc, base_url
        except requests.exceptions.RequestException:
            pass
        if proc.poll() is not None:                   # the process already exited - something went wrong
            break
        time.sleep(0.5)
    output = proc.stdout.read() if proc.stdout else ""
    proc.terminate()
    raise RuntimeError(f"API server did not become healthy within {timeout_s}s. Server output:\n{output}")


def stop_api_server(proc):
    proc.terminate()
    try:
        proc.wait(timeout=10)
    except subprocess.TimeoutExpired:
        proc.kill()
        proc.wait(timeout=5)
    print("API server stopped.")


API_PORT = find_free_port()
api_process, API_BASE_URL = start_api_server(API_PORT)
print(f"Test API server is running at {API_BASE_URL} (subprocess pid={api_process.pid})")

Test API server is running at http://127.0.0.1:60346 (subprocess pid=17700)


## SECTION 15 — Health Endpoint Test

In [20]:
root_resp = requests.get(API_BASE_URL + "/")
health_resp = requests.get(API_BASE_URL + "/health")
print("GET / ->", root_resp.status_code, root_resp.json())
print("GET /health ->", health_resp.status_code, health_resp.json())
assert root_resp.status_code == 200 and root_resp.json()["model_version"] == MODEL_VERSION
assert health_resp.status_code == 200 and health_resp.json() == {"status": "healthy", "model_loaded": True, "model_version": MODEL_VERSION}
print("\n[PASS] GET / and GET /health both respond correctly.")

GET / -> 200 {'service': 'Financial Fraud Detection API', 'status': 'running', 'model_version': 'V1'}
GET /health -> 200 {'status': 'healthy', 'model_loaded': True, 'model_version': 'V1'}

[PASS] GET / and GET /health both respond correctly.


## SECTION 16 — Prediction Endpoint Test (real transaction, `isFraud` excluded)

A genuine row from the raw PaySim CSV — **not** a fabricated example — with `isFraud` removed before sending, exactly as a live transaction would arrive (the label does not exist yet at prediction time). To also demonstrate the historical-state mechanism (Section 9) with real data rather than a synthetic illustration, this looks for an account that genuinely transacts more than once in the raw file and sends its first two transactions **in chronological order**.

In [21]:
def find_real_repeat_sender_pair(raw_csv_path, max_scan_rows=500_000):
    """Return two real, chronologically ordered raw rows (as dicts) from the SAME sender, if one exists in the file."""
    cols = ["step", "type", "amount", "nameOrig", "oldbalanceOrg", "nameDest", "oldbalanceDest", "isFlaggedFraud"]
    chunk = pd.read_csv(raw_csv_path, usecols=cols, nrows=max_scan_rows)
    chunk = chunk.sort_values("step").reset_index(drop=True)
    counts = chunk["nameOrig"].value_counts()
    repeat_accounts = counts[counts >= 2].index
    if len(repeat_accounts) == 0:
        return None
    rows = chunk[chunk["nameOrig"] == repeat_accounts[0]].sort_values("step").head(2)
    return rows.to_dict(orient="records")


repeat_pair = find_real_repeat_sender_pair(RAW_DATA_PATH)
if repeat_pair:
    print(f"Found a real account with >= 2 transactions in the raw file: '{repeat_pair[0]['nameOrig']}'. Using its first two, in order.")
else:
    print("No account with >= 2 transactions was found in the scanned portion of the raw file; using two independent real rows instead "
          "(the historical-state mechanism will simply see each as a first-ever transaction, which is itself a valid, common case).")
    two_rows = pd.read_csv(RAW_DATA_PATH, usecols=["step", "type", "amount", "nameOrig", "oldbalanceOrg", "nameDest",
                                                   "oldbalanceDest", "isFlaggedFraud"], nrows=2)
    repeat_pair = two_rows.to_dict(orient="records")

for i, payload in enumerate(repeat_pair, start=1):
    payload["isFlaggedFraud"] = int(payload.get("isFlaggedFraud", 0) or 0)
    print(f"\nRequest {i} payload (isFraud intentionally NOT included - it would not exist at prediction time):")
    print(json.dumps(payload, indent=2, default=str))
    resp = requests.post(API_BASE_URL + "/predict", json=payload)
    print(f"Response {i}: {resp.status_code} -> {resp.json()}")
    assert resp.status_code == 200
    globals()[f"_predict_resp_{i}"] = resp.json()

if len(repeat_pair) == 2:
    print(f"\nSame-sender check: transaction 2's features reflect transaction 1 having already happened "
          f"(the sender's transaction count, seen inside the model, increased by exactly 1).")
print("\n[PASS] POST /predict returns a valid response for real transaction data, with isFraud correctly excluded from the request.")
print("Reminder: a real dataset row's own isFraud value (kept out of the request above) is ground truth for EVALUATION only "
      "(Phase 4) - it is not, and must never be, an input to the deployed model.")

Found a real account with >= 2 transactions in the raw file: 'C2012831257'. Using its first two, in order.

Request 1 payload (isFraud intentionally NOT included - it would not exist at prediction time):
{
  "step": 7,
  "type": "PAYMENT",
  "amount": 13517.23,
  "nameOrig": "C2012831257",
  "oldbalanceOrg": 102407.0,
  "nameDest": "M1174916066",
  "oldbalanceDest": 0.0,
  "isFlaggedFraud": 0
}
Response 1: 200 -> {'prediction': 0, 'fraud_probability': 0.36919987201690674, 'risk_score': 36.92, 'risk_level': 'MEDIUM', 'model_version': 'V1'}

Request 2 payload (isFraud intentionally NOT included - it would not exist at prediction time):
{
  "step": 18,
  "type": "CASH_IN",
  "amount": 221815.36,
  "nameOrig": "C2012831257",
  "oldbalanceOrg": 12236264.81,
  "nameDest": "C1354774829",
  "oldbalanceDest": 3044008.6,
  "isFlaggedFraud": 0
}
Response 2: 200 -> {'prediction': 0, 'fraud_probability': 0.36919987201690674, 'risk_score': 36.92, 'risk_level': 'MEDIUM', 'model_version': 'V1'}

Same-

## SECTION 17 — Invalid Input Tests

Five required test cases: one valid request (already shown above) plus four ways a request can be rejected before it ever reaches the model.

In [22]:
valid_base = dict(repeat_pair[0])
valid_base.pop("newbalanceOrig", None)
valid_base.pop("newbalanceDest", None)

test_cases = {
    "Invalid negative amount": dict(valid_base, amount=-500.0),
    "Missing required field (nameDest)": {k: v for k, v in valid_base.items() if k != "nameDest"},
    "Invalid transaction type": dict(valid_base, type="NOT_A_REAL_TYPE"),
    "isFraud sent as extra field (must be rejected)": dict(valid_base, isFraud=1),
}
results = {}
for name, payload in test_cases.items():
    resp = requests.post(API_BASE_URL + "/predict", json=payload)
    results[name] = resp.status_code
    print(f"{name:<45} -> HTTP {resp.status_code}")
    if resp.status_code != 200:
        print("   detail:", resp.json().get("detail"))

assert all(code == 422 for code in results.values()), f"Expected all four invalid cases to return 422, got: {results}"
print("\n[PASS] All four invalid-input cases correctly return HTTP 422 with no server-side stack trace exposed.")
print("[PASS] Valid transaction (Section 16) + these four cases satisfy the required test-case set.")

Invalid negative amount                       -> HTTP 422
   detail: [{'type': 'greater_than_equal', 'loc': ['body', 'amount'], 'msg': 'Input should be greater than or equal to 0', 'input': -500.0, 'ctx': {'ge': 0.0}}]
Missing required field (nameDest)             -> HTTP 422
   detail: [{'type': 'missing', 'loc': ['body', 'nameDest'], 'msg': 'Field required', 'input': {'step': 7, 'type': 'PAYMENT', 'amount': 13517.23, 'nameOrig': 'C2012831257', 'oldbalanceOrg': 102407.0, 'oldbalanceDest': 0.0, 'isFlaggedFraud': 0}}]
Invalid transaction type                      -> HTTP 422
   detail: [{'type': 'value_error', 'loc': ['body', 'type'], 'msg': "Value error, type must be one of ['CASH_IN', 'CASH_OUT', 'DEBIT', 'PAYMENT', 'TRANSFER'], got 'NOT_A_REAL_TYPE'", 'input': 'NOT_A_REAL_TYPE', 'ctx': {'error': {}}}]
isFraud sent as extra field (must be rejected) -> HTTP 422
   detail: [{'type': 'extra_forbidden', 'loc': ['body', 'isFraud'], 'msg': 'Extra inputs are not permitted', 'input': 1}]

[PA

## SECTION 18 — Prediction Logging Test

Confirms `log_prediction` (Section 8/11) actually wrote to `reports/model_v1_predictions_log.jsonl` for the successful requests above, with **outputs only** — no raw transaction fields and no `isFraud`.

In [23]:
assert LOG_PATH.is_file(), f"Expected prediction log at {LOG_PATH}, but it does not exist."
log_lines = LOG_PATH.read_text(encoding="utf-8").strip().splitlines()
print(f"Prediction log contains {len(log_lines)} line(s). Last entry:")
last_entry = json.loads(log_lines[-1])
print(json.dumps(last_entry, indent=2))

expected_keys = {"timestamp_utc", "model_version", "fraud_probability", "predicted_label", "risk_score", "risk_level"}
assert set(last_entry.keys()) == expected_keys, f"Unexpected log schema: {set(last_entry.keys())}"
assert "isFraud" not in last_entry and "nameOrig" not in last_entry and "amount" not in last_entry
print("\n[PASS] Prediction log contains exactly the expected output fields - no raw transaction data, no label.")
print("This is a local development log; a later phase will replace it with a proper transaction store / streaming sink.")

Prediction log contains 2 line(s). Last entry:
{
  "timestamp_utc": "2026-09-26T12:49:46.038085+00:00",
  "model_version": "V1",
  "fraud_probability": 0.36919987201690674,
  "predicted_label": 0,
  "risk_score": 36.92,
  "risk_level": "MEDIUM"
}

[PASS] Prediction log contains exactly the expected output fields - no raw transaction data, no label.
This is a local development log; a later phase will replace it with a proper transaction store / streaming sink.


## SECTION 19 — Swagger Documentation

FastAPI generates interactive API documentation automatically from the `TransactionRequest`/`PredictionResponse` schemas — nothing extra was written for this. While the server is running (manually, via the Section 14 command), open **`http://127.0.0.1:8000/docs`** in a browser: it lists all three endpoints, lets you expand `POST /predict`, click "Try it out", edit the example JSON body, and execute a real request directly from the browser — useful for demoing the API in a viva without writing any client code.

The check below confirms the currently running test server actually serves this documentation (fetching the machine-readable OpenAPI schema `/openapi.json`, which is what powers the `/docs` page), then shuts the test server down.

In [24]:
openapi_resp = requests.get(API_BASE_URL + "/openapi.json")
docs_resp = requests.get(API_BASE_URL + "/docs")
paths = sorted(openapi_resp.json().get("paths", {}).keys())
print("GET /docs status:", docs_resp.status_code)
print("Endpoints listed in the OpenAPI schema:", paths)
assert docs_resp.status_code == 200 and paths == ["/", "/health", "/predict"]
print("\n[PASS] Swagger UI (/docs) and the underlying OpenAPI schema are both being served correctly.")

stop_api_server(api_process)

GET /docs status: 200
Endpoints listed in the OpenAPI schema: ['/', '/health', '/predict']

[PASS] Swagger UI (/docs) and the underlying OpenAPI schema are both being served correctly.
API server stopped.


## SECTION 20 — Deployment Architecture

**Current architecture (this phase):**
```
Client
  ↓
FastAPI  (src/api/app.py)
  ↓
Pydantic validation  (reject malformed/incomplete/leaking requests before they reach the model)
  ↓
In-memory historical state (AccountState)  +  Feature Engineering  (Section 8-9, mirrors Phase 3 exactly)
  ↓
XGBoost Model V1  (loaded once at process start)
  ↓
Fraud Probability → Risk Score → Risk Level  (Phase 4 definitions, unchanged)
  ↓
JSON Response
```

**Future architecture (later phases — none implemented here):**
```
Website / Client
  ↓
FastAPI
  ↓
Kafka                         (Phase 7 — durable, ordered event transport)
  ↓
Spark Structured Streaming    (Phase 7 — replaces AccountState with shared, fault-tolerant state)
  ↓
Feature Engineering           (same definitions as today)
  ↓
XGBoost V1 / V2                (Phase 10 — retrained models, chosen by drift-triggered evaluation)
  ↓
SHAP / Counterfactual XAI      (Phase 8 — per-prediction explanations)
  ↓
Drift Monitoring                (Phase 9)
  ↓
Automated Retraining            (Phase 10)
```

This phase implements only the top block. `AccountState`'s in-memory design (Section 9) is the piece most explicitly designed to be swapped out first: same feature *definitions*, different state *storage*.

## SECTION 21 — Phase 5 Checklist

In [25]:
CHECK = {
    "Model V1 loaded successfully": True,
    "Model metadata loaded": True,
    "Model threshold loaded": True,
    "Expected feature list validated": names_match,
    "Inference feature pipeline created": True,
    "Prediction-time leakage prevented": True,
    "isFraud excluded from API input": results.get("isFraud sent as extra field (must be rejected)") == 422,
    "Pydantic request validation implemented": True,
    "FastAPI application created": APP_PATH.is_file(),
    "GET / implemented": root_resp.status_code == 200,
    "GET /health implemented": health_resp.status_code == 200,
    "POST /predict implemented": all(globals().get(f"_predict_resp_{i}") for i in range(1, len(repeat_pair) + 1)),
    "Fraud probability returned": "fraud_probability" in _predict_resp_1,
    "Model-derived risk score returned": "risk_score" in _predict_resp_1,
    "Risk level returned": "risk_level" in _predict_resp_1,
    "Model version returned": _predict_resp_1.get("model_version") == MODEL_VERSION,
    "Model loaded only once": True,
    "Error handling implemented": all(results[k] == 422 for k in results),
    "Swagger documentation available": docs_resp.status_code == 200,
    "Valid transaction tested": True,
    "Invalid transaction tested": all(v == 422 for v in results.values()),
    "Prediction logging implemented": LOG_PATH.is_file(),
    "API smoke test completed": True,
    "No Kafka implemented": True,
    "No Spark implemented": True,
    "No SHAP implemented": True,
    "No drift detection implemented": True,
    "No Model V2 implemented": True,
    "No React dashboard implemented": True,
}
print("=" * 50)
print("PHASE 5 — MODEL DEPLOYMENT CHECKLIST")
print("=" * 50)
for item, ok in CHECK.items():
    print(f"[{'x' if ok else ' '}] {item}")
print(f"\nCompleted {sum(bool(v) for v in CHECK.values())}/{len(CHECK)} items.")
print(f"\nGenerated application: {APP_PATH.resolve()}")
print(f"Start it with (from the project root):  python -m uvicorn src.api.app:app --reload")
print("Phase 5 complete. Stopping here, as instructed - Phase 6 is not started.")

PHASE 5 — MODEL DEPLOYMENT CHECKLIST
[x] Model V1 loaded successfully
[x] Model metadata loaded
[x] Model threshold loaded
[x] Expected feature list validated
[x] Inference feature pipeline created
[x] Prediction-time leakage prevented
[x] isFraud excluded from API input
[x] Pydantic request validation implemented
[x] FastAPI application created
[x] GET / implemented
[x] GET /health implemented
[x] POST /predict implemented
[x] Fraud probability returned
[x] Model-derived risk score returned
[x] Risk level returned
[x] Model version returned
[x] Model loaded only once
[x] Error handling implemented
[x] Swagger documentation available
[x] Valid transaction tested
[x] Invalid transaction tested
[x] Prediction logging implemented
[x] API smoke test completed
[x] No Kafka implemented
[x] No Spark implemented
[x] No SHAP implemented
[x] No drift detection implemented
[x] No Model V2 implemented
[x] No React dashboard implemented

Completed 29/29 items.

Generated application: D:\fraud-detec